# Predicting whether a loan will default (Charged Off) vs be full repaid, using loan application and performance data from LendingClub 

Lenders lose money when borrowers fail to repay, so estimating the probability of default (PD) before approving a loan is a core part of credit risk management. This project uses historical LendingClub loan data (2007–2019) to predict whether a loan will be Fully Paid or Charged Off, based on information available when the loan was issued, such as loan amount, interest rate, income, debt-to-income ratio and credit history. Because defaults are much rarer than repayments, the project addresses class imbalance and evaluates models with precision, recall and ROC-AUC instead of accuracy. It also considers the different costs of approving a loan that defaults versus rejecting one that would have been repaid, to choose a decision threshold a lender could use in practice.

## Intial Setup & inspect structure
Target: default = 1 for Charged Off and Default, Fully Paid = 0. Excluded loans with no final outcome yet (Current, Late, In Grace Period) and early loans issued under a superseded credit policy. Result: 1,385,902 loans, about 19.8% defaulted.

In [4]:
import gc 
import pandas as pd

df_raw = pd.read_csv("../data/raw/appl_accepted_20072019Q3.csv", low_memory=False)
print(df_raw['loan_status'].value_counts())

df = df_raw[df_raw['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()

del df_raw
gc.collect()

print(df.shape)
df.to_parquet("../data/processed/loans_resolved.parquet", index=False)

loan_status
Current                                                1217855
Fully Paid                                             1111884
Charged Off                                             273973
Late (31-120 days)                                       27067
In Grace Period                                          11386
Late (16-30 days)                                         5558
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     45
Name: count, dtype: int64
(1385902, 150)


In [6]:
df = pd.read_parquet("../data/processed/loans_resolved.parquet")
df.info(verbose=True, show_counts=True)

# 0 = Fully Paid, 1 = Charged Off or Default
df['default'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)
print(df['default'].value_counts(normalize=True))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1385902 entries, 0 to 1385901
Data columns (total 150 columns):
 #    Column                                      Non-Null Count    Dtype  
---   ------                                      --------------    -----  
 0    id                                          1385902 non-null  object 
 1    member_id                                   0 non-null        float64
 2    loan_amnt                                   1385902 non-null  float64
 3    funded_amnt                                 1385902 non-null  float64
 4    funded_amnt_inv                             1385902 non-null  float64
 5    term                                        1385902 non-null  object 
 6    int_rate                                    1385902 non-null  object 
 7    installment                                 1385902 non-null  float64
 8    grade                                       1385902 non-null  object 
 9    sub_grade                                   

### Missingness check

In [7]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
print(missing[missing > 0].to_string())

member_id                                     100.000000
next_pymnt_d                                   99.996753
orig_projected_additional_accrued_interest     99.728624
hardship_payoff_balance_amount                 99.584675
hardship_last_payment_amount                   99.584675
hardship_reason                                99.584675
hardship_type                                  99.584675
hardship_status                                99.584675
hardship_amount                                99.584675
deferral_term                                  99.584675
hardship_end_date                              99.584675
hardship_dpd                                   99.584675
hardship_loan_status                           99.584675
hardship_length                                99.584675
payment_plan_start_date                        99.584675
hardship_start_date                            99.584675
sec_app_mths_since_last_major_derog            99.412946
sec_app_revol_util             